# **Camada Silver**

- Importando a base de dados resultante da camada bronze

In [7]:
import pandas as pd
import numpy as np

In [11]:
df_bronze = pd.read_csv('../Data/Bronze/tabela_bronze.csv')

- Criando uma cópia para tratamento

In [12]:
df = df_bronze.copy(deep=True)

- Dimensão Inicial

In [13]:
df.shape

(5634, 23)

- Análise Exploratória Inicial

In [14]:
def analise_inicial(df):
    print("Informações do DataFrame:")
    display(df.info())
    print("\nEstatísticas descritivas:")
    display(df.describe())
    print("\nValores nulos por coluna:")
    display(df.isnull().sum())
    print("\nValores Duplicados por coluna:")
    display(df.duplicated().sum())
    print("\nNúmero de linhas e colunas:")
    display(df.shape)

In [15]:
analise_inicial(df)

Informações do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   str    
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-

None


Estatísticas descritivas:


,SeniorCitizen,MonthlyCharges,TotalCharges,MonthlyIncome
count,5634.000000,5634.000000,5356.000000,5634.000000
mean,0.164714,64.609523,2271.883150,5104.879304
std,0.370955,30.152186,2253.784765,2839.230500
min,0.000000,18.400000,18.800000,100.000000
25%,0.000000,35.162500,412.900000,2693.750000
50%,0.000000,70.300000,1390.725000,5079.500000
75%,0.000000,89.800000,3729.750000,7574.750000
max,1.000000,118.650000,8684.800000,9993.000000



Valores nulos por coluna:


customerID            0
gender                0
SeniorCitizen         0
Partner               0
Dependents            0
tenure                0
PhoneService          0
MultipleLines         0
InternetService       0
OnlineSecurity        0
OnlineBackup          0
DeviceProtection      0
TechSupport           0
StreamingTV           0
StreamingMovies       0
Contract              0
PaperlessBilling      0
PaymentMethod         0
MonthlyCharges        0
TotalCharges        278
Churn                 0
CustomerFeedback      0
MonthlyIncome         0
dtype: int64


Valores Duplicados por coluna:


np.int64(0)


Número de linhas e colunas:


(5634, 23)

## **Limpeza dos Dados**

In [16]:
# Limpar espaços nos textos e transformar campos vazios em nulos
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].astype('string').str.strip()

colunas_tratamento = [c for c in df.columns if c != 'CustomerFeedback']

for coluna in colunas_tratamento:
    valores = df[coluna].astype('string').str.casefold()
    mascara_ausentes = valores.isin([
        '', 'null', 'unknown', 'unknow', 'n/a'
    ])

    df.loc[mascara_ausentes, coluna] = pd.NA

df['customerID'] = df['customerID'].str.upper()

print("Dimensão após limpeza:", df.shape)
print("\nValores ausentes:")
display(df.isna().sum().sort_values(ascending=True))

Dimensão após limpeza: (5634, 23)

Valores ausentes:


customerID            0
gender                0
SeniorCitizen         0
Partner               0
Dependents            0
PhoneService          0
MultipleLines         0
InternetService       0
TechSupport           0
OnlineSecurity        0
OnlineBackup          0
DeviceProtection      0
StreamingMovies       0
StreamingTV           0
Contract              0
PaperlessBilling      0
MonthlyIncome         0
PaymentMethod         0
MonthlyCharges        0
Churn                 0
CustomerFeedback      0
tenure               19
TotalCharges        278
dtype: int64

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   string 
 1   gender            5634 non-null   string 
 2   SeniorCitizen     5634 non-null   float64
 3   Partner           5634 non-null   string 
 4   Dependents        5634 non-null   string 
 5   tenure            5615 non-null   string 
 6   PhoneService      5634 non-null   string 
 7   MultipleLines     5634 non-null   string 
 8   InternetService   5634 non-null   string 
 9   OnlineSecurity    5634 non-null   string 
 10  OnlineBackup      5634 non-null   string 
 11  DeviceProtection  5634 non-null   string 
 12  TechSupport       5634 non-null   string 
 13  StreamingTV       5634 non-null   string 
 14  StreamingMovies   5634 non-null   string 
 15  Contract          5634 non-null   string 
 16  PaperlessBilling  5634 non-null   string 
 17  Paymen

Tratamento anterior:
d'''f = df.replace(['', 'null', 'NULL', 'unknown'], pd.NA)
df['customerID'] = df['customerID'].str.upper()

ID válido: quatro números, hífen e cinco letras (exemplo: 4578-PHJYZ)
id_valido = df['customerID'].str.fullmatch(r'[0-9]{4}-[A-Z]{5}', na=False)

df_linhas_invalidas = df[~id_valido].copy()
df = df[id_valido].copy()

Remover somente linhas totalmente iguais
df = df.drop_duplicates().reset_index(drop=True)
print('Clientes:', len(df))
print('Linhas separadas para conferir:', len(df_linhas_invalidas))'''

## **Tradução dos Dados**

In [18]:
# Traduções usadas em mais de uma coluna
sim_nao = {
    'yes': 'Sim',
    'no': 'Não'
}

servico_internet = {
    'yes': 'Sim',
    'no': 'Não',
    'no internet service': 'Sem serviço de internet'
}

# Tradução de cada coluna
traducao_valores = {
    'gender': {
        'male': 'Masculino',
        'female': 'Feminino'
    },
    'Partner': sim_nao,
    'Dependents': sim_nao,
    'PhoneService': sim_nao,
    'PaperlessBilling': sim_nao,
    'Churn': sim_nao,

    'MultipleLines': {
        'yes': 'Sim',
        'no': 'Não',
        'no phone service': 'Sem serviço de telefonia'
    },
    'InternetService': {
        'dsl': 'DSL',
        'fiber optic': 'Fibra óptica',
        'no': 'Sem serviço de internet'
    },
    'OnlineSecurity': servico_internet,
    'OnlineBackup': servico_internet,
    'DeviceProtection': servico_internet,
    'TechSupport': servico_internet,
    'StreamingTV': servico_internet,
    'StreamingMovies': servico_internet,

    'Contract': {
        'month-to-month': 'Mensal',
        'one year': 'Um ano',
        'two year': 'Dois anos'
    },
    'PaymentMethod': {
        'electronic check': 'Cheque eletrônico',
        'mailed check': 'Cheque pelo correio',
        'bank transfer (automatic)': 'Transferência bancária automática',
        'credit card (automatic)': 'Cartão de crédito automático'
    }
}

# Colocar os textos em minúsculas e aplicar as traduções
for coluna, mapa in traducao_valores.items():
    df[coluna] = df[coluna].str.lower().replace(mapa)

# Conferir o resultado
df[['gender', 'InternetService', 'Contract', 'PaymentMethod', 'Churn']].head()


,gender,InternetService,Contract,PaymentMethod,Churn
0,Masculino,DSL,Um ano,Cheque eletrônico,Não
1,Masculino,DSL,Um ano,Cheque pelo correio,Sim
2,Feminino,Sem serviço de internet,Um ano,Cheque pelo correio,Não
3,Feminino,DSL,Mensal,Cheque eletrônico,Não
4,Masculino,Fibra óptica,Mensal,Cheque eletrônico,Não


## **Tradução das Colunas**

In [19]:
# Traduzir as colunas de acordo com a descrição da sua base
traducao_colunas = {
    'customerID': 'ID_Cliente', 
    'gender': 'Genero',
    'SeniorCitizen': 'Idoso',
    'Partner': 'Conjuge', 
    'Dependents': 'Dependentes', 
    'tenure': 'Tempo_Assinatura',
    'PhoneService': 'Servico_Telefonia', 
    'MultipleLines': 'Varias_Linhas',
    'InternetService': 'Servico_Internet', 
    'OnlineSecurity': 'Seguranca_Online',
    'OnlineBackup': 'Backup_Online', 
    'DeviceProtection': 'Protecao_Dispositivos',
    'TechSupport': 'Suporte_Tecnico', 
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filmes', 
    'Contract': 'Contrato',
    'PaperlessBilling': 'Faturamento_Eletronico', 
    'PaymentMethod': 'Forma_Pagamento',
    'MonthlyCharges': 'Valores_Mensais', 
    'TotalCharges': 'Valores_Totais',
    'Churn': 'Status_Cancelamento', 
    'CustomerFeedback': 'Feedback_Cliente',
    'MonthlyIncome': 'Renda_Mensal'
}
df = df.rename(columns=traducao_colunas)

- Visualizando o resultado

In [20]:
df.head()

,ID_Cliente,Genero,Idoso,Conjuge,Dependentes,Tempo_Assinatura,Servico_Telefonia,Varias_Linhas,Servico_Internet,Seguranca_Online,...,Streaming_TV,Streaming_Filmes,Contrato,Faturamento_Eletronico,Forma_Pagamento,Valores_Mensais,Valores_Totais,Status_Cancelamento,Feedback_Cliente,Renda_Mensal
0,4578-PHJYZ,Masculino,0.0,Sim,Sim,52,Sim,Não,DSL,Não,...,Sim,Não,Um ano,Sim,Cheque eletrônico,68.75,3482.85,Não,I have been a customer with this internet prov...,6532.0
1,6289-CPNLD,Masculino,0.0,Sim,Sim,33,Sim,Não,DSL,Não,...,Sim,Sim,Um ano,Sim,Cheque pelo correio,73.90,2405.05,Sim,I've been a customer with this company for ove...,7634.0
2,2682-KEVRP,Feminino,1.0,Não,Não,22,Sim,Não,Sem serviço de internet,Sem serviço de internet,...,Sem serviço de internet,Sem serviço de internet,Um ano,Sim,Cheque pelo correio,20.05,417.00,Não,I have been a customer with this company for 2...,3628.0
3,5697-GOMBF,Feminino,1.0,Sim,Sim,28,Não,Sem serviço de telefonia,DSL,Não,...,Sim,Não,Mensal,Sim,Cheque eletrônico,35.90,973.65,Não,I have been a customer with this internet prov...,7851.0
4,9717-QEBGU,Masculino,0.0,Não,Não,2,Sim,Não,Fibra óptica,Não,...,Não,Não,Mensal,Sim,Cheque eletrônico,70.30,144.00,Não,I have been using the fiber optic internet ser...,1691.0


- Escolha de Dados: Decidimos manter a coluna de "Feedback_Cliente" na língua original, por conta que os modelos de análise de sentimento que vão ser utilizados (Ou não no futuro), são melhores aplicados quando ;estão na língua inglesa, por definição de negócio esse não é o foco atual, porém pode ser uma coluna poderosa para agregação da análise!

## **Tipagem dos dados**

In [21]:
# Números: tempo em meses, cobranças e renda
colunas_numericas = ['Tempo_Assinatura', 'Idoso', 'Valores_Mensais', 'Valores_Totais', 'Renda_Mensal']
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='raise')

df = df.astype({
    'Tempo_Assinatura': 'Int64',
    'Valores_Mensais': 'Float64',
    'Valores_Totais': 'Float64',
    'Renda_Mensal': 'Float64'
})

df['Idoso'] = df['Idoso'].astype('boolean')

colunas_booleanas = ['Conjuge', 'Dependentes', 'Servico_Telefonia',
                     'Faturamento_Eletronico', 'Status_Cancelamento']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].replace({'Sim': True, 'Não': False}).astype('boolean')

colunas_categoricas = ['Genero', 'Servico_Internet', 'Varias_Linhas',
                      'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivos',
                      'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes',
                      'Contrato', 'Forma_Pagamento']
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype('category')

- Olhando a nova tipagem dos dados

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   ID_Cliente              5634 non-null   string  
 1   Genero                  5634 non-null   category
 2   Idoso                   5634 non-null   boolean 
 3   Conjuge                 5634 non-null   boolean 
 4   Dependentes             5634 non-null   boolean 
 5   Tempo_Assinatura        5615 non-null   Int64   
 6   Servico_Telefonia       5634 non-null   boolean 
 7   Varias_Linhas           5634 non-null   category
 8   Servico_Internet        5634 non-null   category
 9   Seguranca_Online        5634 non-null   category
 10  Backup_Online           5634 non-null   category
 11  Protecao_Dispositivos   5634 non-null   category
 12  Suporte_Tecnico         5634 non-null   category
 13  Streaming_TV            5634 non-null   category
 14  Streaming_Filmes        5634 non-nu

- Visualizando a tabela de números nulos

In [24]:
tabela_nulos = pd.DataFrame({
    'Total Nulos': df.isnull().sum(),
    'Porcentagem (%)': (df.isnull().mean() * 100).round(2)
})
tabela_nulos

,Total Nulos,Porcentagem (%)
ID_Cliente,0,0.00
Genero,0,0.00
Idoso,0,0.00
Conjuge,0,0.00
Dependentes,0,0.00
Tempo_Assinatura,19,0.34
Servico_Telefonia,0,0.00
Varias_Linhas,0,0.00
Servico_Internet,0,0.00
Seguranca_Online,0,0.00


- Verificando se os nulos encontrados na coluna "Valores_Totais" possuem alguma relação ao tempo de assinatura

In [27]:
# Verificar a quantidade e proporção de valores nulos
total_nulos = df["Valores_Totais"].isna().sum()
total_registros = len(df)

print(f"Valores nulos: {total_nulos}")
print(f"Percentual: {total_nulos / total_registros * 100:.2f}%")

# Verificar se os nulos estão relacionados ao tempo de assinatura
analise_nulos = df.groupby(
    df["Valores_Totais"].isna(),
    observed=True
).agg(
    Quantidade=("ID_Cliente", "count"),
    Tempo_Medio=("Tempo_Assinatura", "mean"),
    Valor_Mensal_Medio=("Valores_Mensais", "mean")
)

analise_nulos.index = ["Não nulos", "Nulos"]

display(analise_nulos)

Valores nulos: 278
Percentual: 4.93%


,Quantidade,Tempo_Medio,Valor_Mensal_Medio
Não nulos,5356,32.441551,64.471107
Nulos,278,29.696751,67.276259


- Verificação de registros com assinatura igual a zero

In [33]:
display(
    df[
        df["Tempo_Assinatura"] == 0
    ][
        [
            "ID_Cliente",
            "Tempo_Assinatura",
            "Valores_Mensais",
            "Valores_Totais",
            "Status_Cancelamento"
        ]
    ]
)

,ID_Cliente,Tempo_Assinatura,Valores_Mensais,Valores_Totais,Status_Cancelamento
1409,2520-SGTTA,0,20.0,<NA>,False
2448,2923-ARZLG,0,19.7,<NA>,False
2526,4472-LVYGI,0,52.55,<NA>,False
3306,1371-DWPAZ,0,56.05,<NA>,False
3483,3213-VVOLG,0,25.35,<NA>,False
4726,7644-OMVMY,0,19.85,<NA>,False
4851,3115-CZMZD,0,20.25,<NA>,False
5304,2775-SEFEE,0,61.9,<NA>,False
5378,4367-NUYAO,0,25.75,<NA>,False
5552,4075-WKNIU,0,73.35,<NA>,False


Os valores ausentes da variável Valores_Totais foram preservados na camada Silver, sem aplicação imediata de técnicas de imputação estatística. Essa decisão foi adotada para evitar a introdução de valores artificiais e preservar a distribuição original dos dados. A necessidade de tratamento será avaliada individualmente na camada Gold, conforme os requisitos de cada análise. Eventuais correções serão realizadas somente quando houver justificativa fundamentada nos dados e nas regras de negócio.

In [34]:
df.to_csv("../Data/Silver/tabela_silver.csv", index=False, encoding='utf-8-sig')